# Temporal Analysis

How delays distribute across time dimensions: hour of day, weekday, month, season and full year.

## Setup

In [ ]:
from zh_tram_flow.notebook import *
import zh_tram_flow.analytics.temporal as an

TRAIN, TEST, lf = setup_analysis("03_analysis_3-temporal")
lf_all   = pl.concat([pl.scan_parquet(TRAIN), pl.scan_parquet(TEST)])
lf_delay = lf_all.filter(pl.col("canceled") == False)
lf_clean = (
    lf_all
    .filter(pl.col("canceled") == False)
    .filter(~((pl.col("operating_date").dt.year() == 2025) & (pl.col("operating_date").dt.month() >= 11)))
    .filter(pl.col("line_name") != "E")
    .filter(pl.col("stop_sequence") > 1)
)

%load_ext autoreload
%autoreload 2

## Hour of Day

Ø `arrival_delay` pro Stunde — Rush-Hour-Muster und Nachtbetrieb. Rechts: Volumen zeigt wann die Daten dünn werden.

In [ ]:
an.plot_hour_of_day(lf_delay, cfg)

In [ ]:
show_df(an.table_hour_of_day(lf_delay))

**Beobachtung:** Das Tagesrhythmus-Chart zeigt ein überraschendes Muster: Kein klassischer symmetrischer Doppel-Peak (Morgen/Abend), sondern ein **asymmetrisches Anstiegsprofil**.

**Ø Delay nach Tageszeit (Ø gesamt ≈55s):**
- Morgen 7h: **48.9s** (leicht *unter* Durchschnitt — kein eigentlicher Peak)
- Morgen 8h: 57.3s (knapp über Durchschnitt)
- Nachmittag 14–17h: 58–65s (kontinuierlich steigend)
- Abend 17h: **65.2s** — erster klarer Peak
- Abend 21h: **67.9s** — absoluter Tages-Peak
- Abend 22h: 64.2s (Abfall)

**Was das bedeutet:** Der Morgenrush (~7–9h) ist im Delay-Signal schwach ausgeprägt — kaum höher als Off-Peak-Werte. Das überrascht, da das subjektive Überfüllungsgefühl morgens gross ist. Mögliche Erklärungen: Fahrgäste sind morgens pünktlich (Arbeitsbeginn), Trams fahren eher nach Fahrplan. Ab 14h steigt der Delay monoton an und kulminiert bei 21h.

**Warum 21h-Spike?** Nicht Kaskaden (dann müsste die Kurve ab 17h kontinuierlich steigen ohne Delle). Sondern **Veranstaltungseffekt**: Konzerte, Fussballspiele und Events enden 20–22h → Abreisewelle → überfüllte Trams → erhöhte Haltezeiten. Konsistent mit F-EVNT-03. Betroffen: L11 (Hallenstadion, Messe), L13/17 (Albisgütli/Letzigrund).

**Warum 2–3h klein?** Sehr geringes Datenvolumen. 3h=478 Halte gesamt, 2h=11'548 — statistisch wenig belastbar. Zudem: Nachtlinien nur Fr/Sa→Sa/So und Feiertage. → `hour` als Feature; Interaktion `hour × has_event` relevant für den 21h-Effekt.

## Day of Week

Average delay per weekday (0=Mon … 6=Sun) — weekend vs weekday patterns.

In [ ]:
an.plot_day_of_week(lf_delay, cfg)

In [ ]:
show_df(an.table_day_of_week(lf_delay))

**Beobachtung:** **Donnerstag zeigt die höchste durchschnittliche Verspätung** — nicht Freitag, wie man erwarten könnte.

**Ø Delay und P95 nach Wochentag:**
| Tag | Ø Delay (s) | P95 (s) |
|:---|---:|---:|
| Mo | 52.3 | 172 |
| Di | 57.7 | 186 |
| Mi | 57.9 | 186 |
| **Do** | **60.4** | **194** |
| Fr | 58.1 | 187 |
| Sa | 57.0 | 189 |
| So | 48.4 | 160 |

**Donnerstag ist auf beiden Metriken Spitze** — sowohl im Ø als auch bei P95 (schlechteste 5%). Mo und So sind die besten Tage.

**Warum Donnerstag?** Zwei plausible Erklärungen:
1. **Events-Effekt**: Donnerstag ist in Zürich ein starker Kultur- und Ausgeh-Abend (Konzerte, Messen, Konferenzen). Ein Teil dieser Events fällt gehäuft auf Donnerstage.
2. **Homeoffice-Hypothese**: Wenn Montag und Freitag die häufigsten HO-Tage sind, konzentriert sich Pendlerverkehr auf Di–Do mit Donnerstag als Spitze — plausibel, aber nicht direkt durch VBZ-Daten belegt.

**Wochenende:** Samstag (57.0s) liegt nur leicht unter Werktagsniveau. Sonntag (48.4s) ist deutlich besser — weniger Berufsverkehr und reduzierter Takt im Gleichgewicht. Montag (52.3s) überraschend niedrig — konsistent mit HO-Hypothese.

→ `weekday` als Feature; Interaktion `weekday × hour` prüfen (Donnerstag-Abend-Block).

## Month

Monthly delay averages — seasonal drift visible at month level.

In [ ]:
an.plot_month_seasonality(lf_delay, cfg)

In [ ]:
show_df(an.table_month_seasonality(lf_delay))

**Beobachtung:** Klare **November-Peak-Anomalie** bestätigt: November ist in beiden vollständigen Jahren der Jahreshöchstwert — November 2023=68.9s, November 2024=72.6s. Im Jahresvergleich sehen die Kurvenformen ähnlich aus (gleiche saisonale Struktur), aber 2024 liegt in fast allen Monaten über 2023.

**Monatlicher Ø Delay (bereinigt, Nov/Dez 2025 als Artefakt ausgeschlossen):**
| Monat | 2023 | 2024 | 2025 |
|:---|---:|---:|---:|
| Jan | 48.8 | 51.9 | 49.3 |
| Mär | 53.3 | 60.2 | 53.3 |
| Jun | 57.4 | 60.1 | 58.8 |
| Okt | 60.8 | 57.2 | 60.1 |
| **Nov** | **68.9** | **72.6** | **70.7** |
| Dez | 63.0 | 57.7 | — |

**Warum November?** Kombination aus mehreren Faktoren:
- Herbstlaub auf Gleisen (Leaf Fall Problem): nasses Laub reduziert Haftung, langsamere Einfahrten
- Ende des Herbst-Baustellen-Zyklus: VBZ/Stadt schliessen Gleisbaustellen typischerweise vor Winterfahrplan ab
- Maximale Systembelastung: Schuljahr läuft, kein Ferieneffekt, Dunkelheit und Regen erhöhen MIV-Anteil

**Jahrestrend:** 2024 deutlich höher als 2023 in den Frühlings-/Sommermonaten (Mär–Jun: +6–7s). 2025 (Jan–Okt) liegt zwischen 2023 und 2024 — kein klarer weiterer Anstieg. → `month`, `year` und `is_november` als Features; November-Dummy als Verstärker prüfen.

In [ ]:
section_header("Season & Heatmap")

## Season

Delay by season (1=Winter 2=Spring 3=Summer 4=Fall) — weather and daylight effects.

In [ ]:
an.plot_season_heatmap(lf_delay, cfg)

In [ ]:
show_df(an.table_season(lf_delay))

**Beobachtung:** Die Saisonauswertung bestätigt das Muster aus der Monatsanalyse.

**Saisonaler Ø Delay und OTP:**
| Jahreszeit | Ø Delay (s) | OTP |
|:---|---:|---:|
| Winter | **51.7** | **88.9%** |
| Frühling | 55.6 | 87.3% |
| Sommer | 56.4 | 86.8% |
| **Herbst** | **61.2** | **85.2%** |

**Herbst** ist die eindeutig schlechteste Jahreszeit (61.2s, OTP 85.2%). **Winter** ist überraschenderweise die beste — Ø 9.5s unter Herbst. Mögliche Erklärung: Im Winter reduziert sich der MIV-Anteil (Menschen meiden Autofahren bei Schnee/Eis), was die Strassenkonflikte für Trams verringert und den Effekt von Schnee/Eis auf die Gleise teilweise kompensiert. Frühling und Sommer liegen nah beieinander (55.6 vs. 56.4s).

Die **Heatmap Stunde × Wochentag** macht das Zusammenspiel sichtbar: der Donnerstag/Freitag-Abend-Block (17–21h) zeigt die dunkelsten Felder. Rush-Hour-Muster sind wochentagsübergreifend erkennbar, aber der Abend-Effekt dominiert gegenüber dem Morgen.

→ `season`, `hour`, `weekday` als Features; Interaktion `hour × weekday` als kombiniertes Feature prüfen.

## Full Year

Weekly or monthly rolling delay trend across all three years — long-term drift and anomalies.

In [ ]:
an.plot_full_year_trend(lf_delay, cfg)

In [ ]:
show_df(an.table_full_year_monthly(lf_delay))

**Beobachtung:** Der Rolling-Average-Chart macht die Netzstruktur über alle drei Jahre sichtbar.

**Schulferien-Effekt:** Die grau hinterlegten Schulferienperioden fallen konsistent mit Verspätungs-Tälern zusammen — besonders deutlich bei Sommer- und Herbstferien. Das ist visuell überzeugend: Weniger Schülerverkehr = weniger Trams übervoll = weniger Verzögerungen beim Boarding.

**Strukturelle Befunde:**
- 2024 liegt im Frühling/Sommer **+4–7s** über dem 2023-Niveau — struktureller Anstieg
- 2025 (Jan–Okt) zeigt leichte Stabilisierung gegenüber 2024 — kein weiterer Anstieg
- **Aufwärtstrend moderat:** Das Netz wird nicht dramatisch schlechter, aber liegt strukturell über dem OTP-Sollwert
- November-Peaks in beiden Jahren deutlich sichtbar (→ F-TEMP-05)
- Fahrplanwechsel Dez 2023 (j23→j24): kein scharfer Knick erkennbar — Übergang fliessend

**Einordnung Aufwärtstrend:**
> 2025 war leicht besser als 2024 — das schwächt eine „alarmierenden Trend"-Story. Seriösere Aussage: Das VBZ-Netz läuft stabil nahe seinem OTP-Ziel, hat aber **keinen strukturellen Puffer** bei Sonderereignissen (Schnee, Grossevents, November). Der Schulferien-Dip zeigt: bei reduziertem Druck funktioniert das Netz gut.

## Feature: `gtfs_year`

Netzwerk-Epoche als Feature: `j23` (vor Fahrplanwechsel Dez 2023) vs. `j24_j25` (nach Umbau der Linien 9, 11, 13). Zeigt ob der Strukturbruch in der Zeitreihe einen Sprung erzeugt oder ob die Verspätung kontinuierlich verläuft (F-NET-01, F-NET-03).

In [ ]:
an.plot_gtfs_year_comparison(lf_delay, cfg)

In [ ]:
show_df(an.table_gtfs_year_comparison(lf_delay))

**Beobachtung:** Der `gtfs_year`-Vergleich zeigt einen **minimal kleinen Netzeffekt**: Netzweit steigt der Ø Delay von j23 auf j24_j25 um nur **+0.5s** (55.9s → 56.4s). Die OTP-Differenz beträgt −0.4pp (87.3% → 86.9%).

**Netzweit: j23 vs. j24_j25:**
| GTFS-Epoche | Ø Delay (s) | OTP | N Halte |
|:---|---:|---:|---:|
| j23 | 55.9 | 87.3% | 28.8M |
| j24_j25 | 56.4 | 86.9% | 60.9M |
| **Δ** | **+0.5s** | **−0.4pp** | |

**Umgebaute Linien 9, 11, 13 — Vor/Nach:**
| Linie | j23 (s) | j24_j25 (s) | Δ |
|:---|---:|---:|---:|
| L11 | 65.1 | 70.6 | **+5.5s** |
| L13 | 51.6 | 53.1 | +1.5s |
| L9 | 58.6 | 54.3 | **−4.3s** |

**Kernbefund:** Das `gtfs_year`-Feature erklärt netzweit fast nichts (+0.5s). Auf Linie-Ebene gibt es Unterschiede, aber sie zeigen keine einheitliche Richtung: L11 verschlechtert sich, L9 verbessert sich deutlich. Das ist konsistent mit dem Befund aus dem Network-Notebook (F-NET-04, F-NET-05): der Fahrplanwechsel Dez 2023 ist im Delay-Signal nicht als Bruchpunkt erkennbar.

**Implikation für Modellierung:** `gtfs_year` dürfte im Modell schwachen Beitrag leisten — eher als Zeitvariable denn als Netzstruktur-Feature. `n_stops_line` als kontinuierliche Alternative bleibt prüfenswert (F-NET-03). Der strukturelle j23→j24-Anstieg im Rolling-Average erklärt sich besser durch Saisonalität und Jahrestrend als durch den Netzwechsel.

## Key Findings

→ Vollständige Findings-Tabelle mit Impact und Action in [`03_analysis_0-overview.ipynb`](03_analysis_0-overview.ipynb).

| ID | Finding | Status |
|:---|:---|:---|
| F-TEMP-01 | Kein klassischer Morgenrush-Peak: 7h=48.9s liegt *unter* Ø. Dominantes Muster ist Nachmittag/Abend (14h aufwärts), Peak bei 21h=67.9s (Events-Abreisewelle), starker Abend-Peak 17h=65.2s | done |
| F-TEMP-02 | **Donnerstag** ist kritischster Wochentag: Ø 60.4s, P95=194s — sowohl im Mittel als auch in den Extremwerten Spitze. Montag (52.3s) und Sonntag (48.4s) beste Tage. | done |
| F-TEMP-03 | Donnerstag-Peak vereinbar mit Events-Häufung (Do-Abend) und Homeoffice-Hypothese (Mo/Fr = HO → Do = Verkehrsspitze) — nicht direkt durch Daten belegt, aber plausibel | done |
| F-TEMP-04 | Wochenende: Samstag (57.0s) kaum besser als Werktag; Sonntag (48.4s) deutlich besser — reduzierter Takt und weniger Berufsverkehr | done |
| F-TEMP-05 | **November-Peak-Anomalie bestätigt**: Nov 2023=67.9s, Nov 2024=72.9s — jeweils Jahreshöchstwert; ca. 10–12s über Jahresschnitt der restlichen Monate | done |
| F-TEMP-06 | Saisonales Muster: Herbst=61.2s (schlechteste Jahreszeit, OTP 85.2%), Winter=51.7s (beste, OTP 88.9%) — Winter besser als Herbst trotz Witterung | done |
| F-TEMP-07 | Struktureller Aufwärtstrend: 2024 liegt in den meisten Monaten +4–7s über 2023; 2025 (Jan–Okt) leicht moderater als 2024 — kein weiterer Anstieg sichtbar | done |
| F-TEMP-08 | Schulferien-Täler erkennbar im Rolling-Average — Schulferien-Flag als Feature-Kandidat | done |
| F-TEMP-09 | `gtfs_year`-Feature erklärt netzweit nur +0.5s (j23=55.9s → j24_j25=56.4s) — schwaches Feature-Signal; auf Linie-Ebene keine einheitliche Richtung (L11 +5.5s, L9 −4.3s) | done |